<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# The mathematics of attention

**[The mathematics of attention](https://learning.nextia-ai.com/courses/deep-learning/m08/the-mathematics-of-attention/)** · Deep Learning and Transformers Explained · Module 8 (optional), lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** write attention and multi-head attention from their formulas, check them against PyTorch's own functions, measure why the scores are divided by √d_k, and time the n × n score matrix on your own machine.

| | |
|---|---|
| **Time** | About 30 minutes. Every cell runs in seconds on a free Colab CPU. |
| **Needs** | No account and no data download. Colab and Kaggle have PyTorch. |
| **You should know** | Queries, keys and values, from [Attention step by step](https://learning.nextia-ai.com/courses/deep-learning/m05/attention-step-by-step/), and multi-head attention, from [Transformer blocks](https://learning.nextia-ai.com/courses/deep-learning/m06/transformer-blocks/). |
| **You do not need** | Any earlier notebook of this course. |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m08/the-mathematics-of-attention/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M08-L03-the-mathematics-of-attention/the-mathematics-of-attention-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

## Setup: reference versions

The next cell defines finished versions of the functions that you write below. The notebook uses them only if one of your tasks is not finished yet, so that every later cell still runs; it then prints a one-line note. Try your own version first. Run the cell. It prints nothing.

In [ ]:
# Reference versions, used only as a fallback when a task below is not finished yet.
import math
import torch.nn.functional as F

def reference_attention(Q, K, V, mask=None):
    scores = Q @ K.transpose(-2, -1) / math.sqrt(Q.shape[-1])
    if mask is not None:
        scores = scores + mask
    weights = torch.softmax(scores, dim=-1)
    return weights @ V, weights

class ReferenceMultiHead(torch.nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.d, self.h, self.d_k = d, h, d // h
        self.W_Q, self.W_K, self.W_V, self.W_O = (torch.nn.Linear(d, d) for _ in range(4))

    def split(self, X):
        B, n, _ = X.shape
        return X.view(B, n, self.h, self.d_k).transpose(1, 2)

    def forward(self, Xq, Xkv, mask=None):
        Q, K, V = self.split(self.W_Q(Xq)), self.split(self.W_K(Xkv)), self.split(self.W_V(Xkv))
        O, weights = reference_attention(Q, K, V, mask)
        B, _, n, _ = O.shape
        return self.W_O(O.transpose(1, 2).reshape(B, n, self.d)), weights

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. From X to queries, keys and values

Three tokens, "damaged on arrival", with d = 4 numbers each, and three made-up weight matrices of shape (4, 2). They are chosen so that Q, K and V equal the worked example of the attention lesson.

> **Your turn.** Compute `Q`, `K` and `V` from `X` with one matrix product each, and the scores `S` (all query–key dot products in one product). Predict the shapes first.

In [ ]:
import math
import torch.nn.functional as F

X = torch.tensor([[1.0, 0.0, 0.0, 1.0],      # damaged
                  [0.0, 1.0, 0.0, 0.0],      # on
                  [0.0, 0.0, 1.0, 0.0]])     # arrival
W_Q = torch.tensor([[0.5, 0.5], [0.0, 0.0], [1.0, 0.0], [0.0, 1.0]])
W_K = torch.tensor([[1.0, 0.0], [0.0, 0.5], [0.0, 2.0], [0.0, 0.0]])
W_V = torch.tensor([[1.0, 0.0], [0.0, 0.0], [0.0, 1.0], [1.0, 0.0]])
Q = ...     # (3, 2)
K = ...
V = ...
S = ...     # (3, 3): S[i, j] = query i . key j

In [ ]:
expect_hash('Q', Q.tolist() if isinstance(Q, torch.Tensor) else None, '396926869e1e20ae')

In [ ]:
expect_hash('S', S.tolist() if isinstance(S, torch.Tensor) else None, '038887b426856fd3')

In [ ]:
# The later cells need Q, K and V. If your task is not finished yet, use the reference values.
if not all(isinstance(t, torch.Tensor) for t in (Q, K, V)):
    print("Note: Q, K or V is not finished yet, so the next cells use the reference values.")
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V

Row 1 of `S` holds the scores of "damaged": 0.5, 0.75 and 3.0, the numbers of the attention lesson.

## 2. Write attention from the formula

softmax(Q Kᵀ / √d_k + M) V. The mask M holds 0 where a query may read a key and −∞ where it may not.

> **Your turn.** Complete `attention()`. It returns the output and the weights. Use `K.transpose(-2, -1)`, so that it also works with a batch and heads in front, and `torch.softmax(…, dim=-1)`.

In [ ]:
def attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = ...                 # Q K^T / sqrt(d_k)
    if mask is not None:
        scores = scores + mask   # -inf where a query may not look
    weights = ...                # softmax, row by row
    return weights @ V, weights

Run the next cell. It calls your `attention()` on the three tokens. If your function is not finished yet, it says so and the later cells use the reference version.

In [ ]:
try:
    out, weights = attention(Q, K, V)
except Exception as error:
    print(f"Note: attention() is not finished yet ({type(error).__name__}), so the next cells use the reference version.")
    my_weights = None
    attention = reference_attention
    out, weights = attention(Q, K, V)
else:
    my_weights = weights
print(weights.round(decimals=3))

In [ ]:
expect_hash("the weights of 'damaged'", [round(float(v), 3) for v in my_weights[0]] if my_weights is not None else None, 'e08a4ade652d99d8')

Now the check against PyTorch's own function, on random batches with and without the causal mask. `torch.triu(…, diagonal=1)` keeps the part above the diagonal: that is where the causal mask holds −∞.

Run the cells.

In [ ]:
torch.manual_seed(0)
q, k, v = (torch.randn(3, 7, 16) for _ in range(3))
causal = torch.triu(torch.full((7, 7), float("-inf")), diagonal=1)
diff = max((attention(q, k, v)[0] - F.scaled_dot_product_attention(q, k, v)).abs().max().item(),
           (attention(q, k, v, causal)[0] - F.scaled_dot_product_attention(q, k, v, is_causal=True)).abs().max().item())
print(f"largest difference: {diff:.1e}")

In [ ]:
expect_hash('your attention() against PyTorch', (diff < 1e-6) if my_weights is not None else None, 'b5bea41b6c623f7c')

> **Predict.** With the causal mask, which tokens may "damaged" (the first token) read? What will its weights be?

In [ ]:
causal3 = torch.triu(torch.full((3, 3), float("-inf")), diagonal=1)
print(causal3)
print(attention(Q, K, V, causal3)[1].round(decimals=3))
# A score of 0 is not hidden: softmax turns it into e^0 = 1.
print("scores set to 0:", torch.softmax(torch.tensor([0.354, 0.0, 0.0]), 0).round(decimals=3))

"damaged" reads only itself: 1, 0, 0. With the hidden scores set to 0 instead of −∞, it would still give 0.584 of its weight to the two hidden words.

## 3. Why divide by √d_k: measure it

If every number of q and k is random with mean 0 and variance 1, then Var(q · k) = d_k. Measure it.

> **Your turn.** For each `d_k` in the list, draw 20,000 queries and 20,000 keys with `torch.randn(20000, d_k)`, compute the 20,000 dot products, and store their variance in `variances[d_k]`. Also store the variance after dividing the dot products by √d_k in `scaled[d_k]`.

In [ ]:
torch.manual_seed(0)
variances, scaled = {}, {}
for d_k in (4, 16, 64, 256):
    ...
print(variances)
print(scaled)

In [ ]:
expect_hash("'every variance within 10% of d_k'", (len(variances) == 4 and all(abs(v / d - 1) < 0.1 for d, v in variances.items())) if variances else None, 'b5bea41b6c623f7c')

In [ ]:
expect_hash("'every scaled variance within 10% of 1'", (len(scaled) == 4 and all(abs(v - 1) < 0.1 for v in scaled.values())) if scaled else None, 'b5bea41b6c623f7c')

> **Predict.** At d_k = 256, how much of the weight will the largest of 32 random scores get after softmax, without and with the division?

Run the cell. It averages the largest softmax weight over 2,000 random queries with 32 keys each.

In [ ]:
for d_k in (4, 64, 256):
    raw = torch.einsum("nd,nkd->nk", torch.randn(2000, d_k), torch.randn(2000, 32, d_k))
    print(f"d_k {d_k:4}: largest weight unscaled {torch.softmax(raw, 1).max(1).values.mean():.3f},"
          f" scaled {torch.softmax(raw / math.sqrt(d_k), 1).max(1).values.mean():.3f}")

Without the division, softmax becomes almost a hard choice of one key as d_k grows, and its gradient p(1 − p) gets close to 0. With the division, the rows look alike for every key size. (The lesson measured 0.335 → 0.903 unscaled and about 0.16 scaled; your random draws give similar numbers.)

## 4. Multi-head attention

h heads of size d_k = d / h. One d × d matrix makes all the queries, and `split` cuts its columns into heads: (B, n, d) → (B, h, n, d_k). After attention, the heads are joined again and `W_O` mixes them.

> **Your turn.** Complete `forward()`: split Q, K and V into heads, call `attention()`, then join the heads back into (B, n, d) with `O.transpose(1, 2).reshape(B, n, self.d)` and apply `W_O`.

In [ ]:
class MultiHead(torch.nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.d, self.h, self.d_k = d, h, d // h
        self.W_Q, self.W_K, self.W_V, self.W_O = (torch.nn.Linear(d, d) for _ in range(4))

    def split(self, X):                       # (B, n, d) -> (B, h, n, d_k)
        B, n, _ = X.shape
        return X.view(B, n, self.h, self.d_k).transpose(1, 2)

    def forward(self, Xq, Xkv, mask=None):
        ...
        return out, weights

torch.manual_seed(0)
mine = MultiHead(64, 4)
n_params = sum(p.numel() for p in mine.parameters())
print(n_params, "parameters")

In [ ]:
expect('the parameter count', n_params, 16640)

In [ ]:
# The later cells need a working MultiHead. If forward() is not finished yet, use the reference version.
try:
    with torch.no_grad():
        o, w = mine(torch.zeros(1, 2, 64), torch.zeros(1, 2, 64))
    if tuple(o.shape) != (1, 2, 64) or tuple(w.shape) != (1, 4, 2, 2):
        raise ValueError("wrong shapes")
except Exception as error:
    print(f"Note: MultiHead.forward() is not finished yet ({type(error).__name__}), so the next cells use the reference version.")
    torch.manual_seed(0)
    mine = ReferenceMultiHead(64, 4)
    my_multihead = False
else:
    my_multihead = True

4d² + 4d = 4 × 64² + 4 × 64 = 16,640, for any number of heads.

Now compare with PyTorch's `nn.MultiheadAttention`. It stores W_Q, W_K and W_V stacked in one matrix, `in_proj_weight`, so the next cell copies ours into it. PyTorch's layer takes a True/False causal mask where `True` means "may not read".

Run the cell.

In [ ]:
ref = torch.nn.MultiheadAttention(64, 4, batch_first=True)
with torch.no_grad():
    ref.in_proj_weight.copy_(torch.cat([mine.W_Q.weight, mine.W_K.weight, mine.W_V.weight]))
    ref.in_proj_bias.copy_(torch.cat([mine.W_Q.bias, mine.W_K.bias, mine.W_V.bias]))
    ref.out_proj.weight.copy_(mine.W_O.weight)
    ref.out_proj.bias.copy_(mine.W_O.bias)
    Xb = torch.randn(3, 7, 64)
    future = torch.triu(torch.ones(7, 7, dtype=torch.bool), diagonal=1)
    o1, w1 = mine(Xb, Xb, causal)
    o2, w2 = ref(Xb, Xb, Xb, attn_mask=future, average_attn_weights=False)
mha_diff = max((o1 - o2).abs().max().item(), (w1 - w2).abs().max().item())
print(f"largest difference: {mha_diff:.1e}")
show_shape("weights", w1)

In [ ]:
expect_hash('your MultiHead against nn.MultiheadAttention', (mha_diff < 1e-6) if my_multihead else None, 'b5bea41b6c623f7c')

> **Your turn.** The tiny transformer reads a ticket of 120 tokens with d = 64 and 4 heads. Run `mine` on a zero tensor of shape (1, 120, 64) and store the shape of the attention weights (a tuple) in `weights_shape` and the number of scores in `n_scores`.

In [ ]:
weights_shape = ...
n_scores = ...

In [ ]:
expect_hash('weights_shape', weights_shape if weights_shape is not ... else None, '27f284eb9807bc62')

In [ ]:
expect_hash('n_scores', n_scores if n_scores is not ... else None, '62da24cc616bf24f')

## 5. The cost of n × n, on your machine

One head, d_k = 64. The cell times `attention()` and PyTorch's fused function for growing n (median of 5 passes) and prints the memory of the score matrix. It takes a few seconds.

> **Predict.** When n doubles, how much longer should it take?

Your numbers will not match the lesson's: Colab, Kaggle and laptops differ, and shared machines vary from run to run. Look at the ratios.

In [ ]:
import statistics, time

def median_ms(f, repeats=5):
    f()                                       # warm-up
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        f()
        times.append(time.perf_counter() - t0)
    return statistics.median(times) * 1000

previous = None
with torch.no_grad():
    for n in (512, 1024, 2048, 4096):
        Qn, Kn, Vn = (torch.randn(1, n, 64) for _ in range(3))
        ms = median_ms(lambda: attention(Qn, Kn, Vn))
        fused = median_ms(lambda: F.scaled_dot_product_attention(Qn, Kn, Vn))
        ratio = f"x{ms / previous:.1f}" if previous else ""
        print(f"n {n:5}: scores {n * n * 4 / 1e6:6.1f} MB   attention() {ms:7.2f} ms {ratio:6}  fused {fused:6.2f} ms")
        previous = ms

The memory grows 4 times for every doubling of n. The time grows by close to 4 times too once n is large; at small n, fixed overheads hide it. PyTorch's fused function is faster, but it still does the n² work.

## 6. A mask that means the opposite

`F.scaled_dot_product_attention` also accepts a True/False mask, but there `True` means "may read", the opposite of `nn.MultiheadAttention`. Run the cell and compare the two outputs.

In [ ]:
future3 = torch.triu(torch.ones(3, 3, dtype=torch.bool), diagonal=1)
print("wrong:", F.scaled_dot_product_attention(Q, K, V, attn_mask=future3))
print("right:", F.scaled_dot_product_attention(Q, K, V, attn_mask=~future3))

No error, but the wrong mask lets each token read only the future. A float mask of 0 and −∞, added to the scores, means the same everywhere.

## Next

You wrote attention and multi-head attention from their formulas, checked them against PyTorch, measured the √d_k argument and saw the n² cost. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m08/the-mathematics-of-attention/) has the explorers, the real ticket's heatmaps and the knowledge checks. The next lesson builds the original transformer of 2017 from these pieces.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [The original transformer](https://learning.nextia-ai.com/courses/deep-learning/m08/the-original-transformer/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m08/the-mathematics-of-attention/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The worked-example numbers are made up; every other number in this notebook comes from random draws or from your own machine.